In [1]:
import re
import torch
from torch import nn

In [2]:
# Set random seed
torch.manual_seed(42)

In [3]:
# Define dataset
docs = ["Movies are fun for everyone.",
        "Watching movies is great fun.",
        "Enjoy a great movie today.",
        "Research is interesting and important.",
        "Learning math is very important.",
        "Science discovery is interesting.",
        "Rock is great to listen to.",
        "Listen to music for fun.",
        "Music is fun for everyone.",
        "Listen to folk music!"]

labels = [1, 1, 1, 3, 3, 3, 2, 2, 2, 2]
num_classes = len(set(labels))

In [4]:
# Split document to token (lowercase word)
def tokenize(doc):
    return re.findall(r"\w+", doc.lower())

In [5]:
# Get all the unique token in docs in sorted order
def get_vocabulary(documents):
    # Get all unique token in docs (using a set to store)
    tokens = {token for doc in documents for token in tokenize(doc)}
    # Return dict: Sorted token and assign idx
    return {word: idx for idx, word in enumerate(sorted(tokens))}

In [6]:
vocabulary = get_vocabulary(docs)

In [7]:
# Convert doc to feature vector
def doc_to_bow(doc, vocabulary):
    # Get token in doc
    tokens = set(tokenize(doc))
    # Init bow with size = vocabulary size
    bow = [0] * len(vocabulary)
    # For each token in doc, if vocab contains then update bow index to 1
    for token in tokens:
        if token in vocabulary:
            bow[vocabulary[token]] = 1
    return bow

In [8]:
vectors = torch.tensor(
    [doc_to_bow(doc, vocabulary) for doc in docs],
    dtype=torch.float32
)
labels = torch.tensor(labels, dtype=torch.long) - 1

In [9]:
# Create neural network
input_dim = len(vocabulary)
hidden_dim = 50
output_dim = num_classes


# Input (batch, len(vocab)) -> layer1 (batch, hidden_dim) -> ReLU -> output (batch, num_classes)
class SimpleClassifier(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        return x

In [10]:
# Define a neural network model using Softmax
model = SimpleClassifier(input_dim, hidden_dim, output_dim)

# Define Cross-Entropy Loss + SGD optimizer
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.001)

# Train for 3000 epochs
for step in range(3000):
    optimizer.zero_grad()
    loss = criterion(model(vectors), labels)
    loss.backward()
    optimizer.step()

In [11]:
# Inference
new_docs = [
    "Listening to rock music is fun.",
    "I love science very much."
]
class_names = ["Cinema", "Music", "Science"]

new_doc_vectors = torch.tensor([
    doc_to_bow(new_doc, vocabulary) for new_doc in new_docs],
    dtype=torch.float32
)

with torch.no_grad():
    outputs = model(new_doc_vectors)
    predicted_ids = torch.argmax(outputs, dim=1) + 1

for i, new_doc in enumerate(new_docs):
    print(f'{new_doc}: {class_names[predicted_ids[i].item() - 1]}')

Listening to rock music is fun.: Music
I love science very much.: Science
